# Policy gradient: CartPole

REINFORCE (Williams, 1992) on a pure Idris CartPole environment. The agent
learns a policy that maps observations (cart position, velocity, pole angle,
angular velocity) to actions (push left/right).

REINFORCE is the simplest of the library's RL examples — A2C, PPO, DQN,
Double-DQN, and the tabular methods build on the same rollout-as-loss
pattern (see `packages/idris-ml-examples/src/Example/`). It demonstrates
that idris-ml handles non-standard training loops beyond supervised learning.

**CLI equivalent:** `make example-reinforce` (2000 epochs, converges to 200.0 return)


## Architecture

A simple 2-layer MLP:
```
Linear(4 -> 128) -> Tanh -> Linear(128 -> 2)
```

Output: 2 logits (push left / push right). Action sampled via
`categoricalSample` from log-softmax probabilities.


In [1]:
:t categoricalSample


Ml.Sampler.categoricalSample : List Double -> Double -> Nat


## CartPole environment

The CartPole physics are implemented in pure Idris (the `idris-gym` package,
no gym/gymnasium dependency). Gymnasium-compatible constants: gravity=9.8,
pole length=0.5, force=10.0, dt=0.02. Episode terminates when:
- Cart position |x| > 2.4
- Pole angle |theta| > 12 degrees
- 200 steps reached (maximum return)

The environment is deterministic given the same random seed.


## REINFORCE algorithm

1. Collect a batch of episodes using the current policy
2. For each step: log_prob * (G_t - baseline)
   - `G_t` = discounted return from step t
   - baseline = mean return across batch (variance reduction)
3. Sum losses, backpropagate, update policy

Each step's loss is a scalar `Tensor []` built from the log-softmax of the
policy's logits, so the autograd graph reaches back to the policy weights;
the rollout control flow around it is ordinary Idris.


## Model construction

The policy network builds interactively. The CartPole environment and the
rollout loop are in the compiled example (`src/Example/Reinforce.idr`).


In [2]:
:exec run (do {
  m <- runInitL (the (Init (Seq 4 2 TapeExecutor F64 WithGrad))
                 (do { l1 <- linear {i=4} {o=128}; l2 <- linear {i=128} {o=2};
                       pure (l1 ~~> tanhA ~~> l2 ~~> Nil) }));
  discard m; liftIO1 (putStrLn "policy net built (4 -> 128 -> 2).") })

policy net built (4 -> 128 -> 2).


Training is `fit` with a custom epoch step: roll out a batch of episodes,
compute the REINFORCE loss, `trainStep` it, and thread the linear policy
through. From `Example/Reinforce.idr`:

```idris
opt <- adam cfg.lr ({ clip := NormClip 1.0 } defaultOpts)

(MkBang (epochsDone, _) # trained) <-
  fit {batch = Vect n (List Double)}
       (\m, d => do
          (MkBang (loss, avgRet) # m') <- computeLossBatchedL cfg.gamma m d
          dd <- liftIO1 (do x <- trainStep opt loss; recordReturn metrics avgRet; pure x)
          pure1 (MkBang dd # m'))
       opt (generate (genBatchV n))
       ({ metricsL := readRLMetrics "recent_100" metrics }
          (simpleConfig {model = Policy} cfg.epochs))
       model
```

The loss value is the negative mean episode return; as training progresses
it approaches -200.0 (perfect balance for all 200 timesteps).


## Scaling up

For full convergence (200.0 greedy return):
```bash
make example-reinforce REINFORCE_ARGS="--epochs 2000 --lr 0.001 --batch 10"
```

Convergence claims here follow the repo's multi-seed policy: an RL example
passes only when it converges on ≥ 5 seeds, matched against the PyTorch
reference's pass rate.


## PyTorch comparison

```python
# Standard REINFORCE
probs = F.softmax(model(obs), dim=-1)
action = torch.multinomial(probs, 1)
log_prob = torch.log(probs[action])

# After episode:
loss = -sum(log_prob * (G_t - baseline))
loss.backward()
optimizer.step()
```

In idris-ml, the rollout loop and loss computation are ordinary Idris code;
the tensor operations (`forwardSeq`, the log-softmax, the per-step scalar
losses) go through the C backend and participate in autograd.

See `pytorch/torch_ref/scripts/reinforce.py` for the full reference.


Next: [SeqClassify](seq_classify.ipynb) — 1D convolutions for waveform classification.
